# Notebook 5: Risk Scoring Ablation
This notebook runs ablation and clustering validations on risk scoring metrics.

In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
conda_prefix = r"C:\Users\harsh\anaconda3\envs\raphael-env"
lib_bin = os.path.join(conda_prefix, "Library", "bin")
if os.path.exists(lib_bin):
    if lib_bin not in os.environ["PATH"]:
        os.environ["PATH"] = lib_bin + os.pathsep + os.environ["PATH"]
    if sys.platform == 'win32' and hasattr(os, 'add_dll_directory'):
        try: os.add_dll_directory(lib_bin)
        except: pass
import sqlite3
sys.path.insert(0, os.path.abspath('..'))  # backend root
DB_PATH = os.path.abspath('../data/raphael.db')
if not os.path.exists(DB_PATH):
    DB_PATH = os.path.abspath('data/raphael.db')
assert os.path.exists(DB_PATH), f"DB not found: {DB_PATH}"
conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

PUNE_REGION_ID = conn.execute(
    "SELECT id FROM regions WHERE name='Pune Metropolitan Region'"
).fetchone()[0]
print(f"Pune region ID: {PUNE_REGION_ID}")


In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
conda_prefix = r"C:\Users\harsh\anaconda3\envs\raphael-env"
lib_bin = os.path.join(conda_prefix, "Library", "bin")
if os.path.exists(lib_bin):
    if lib_bin not in os.environ["PATH"]:
        os.environ["PATH"] = lib_bin + os.pathsep + os.environ["PATH"]
    if sys.platform == 'win32' and hasattr(os, 'add_dll_directory'):
        try: os.add_dll_directory(lib_bin)
        except: pass

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'text.color':       '#c9d1d9',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#c9d1d9',
    'ytick.color':      '#c9d1d9',
    'axes.edgecolor':   '#30363d',
    'grid.color':       '#21262d',
    'savefig.facecolor':'#0d1117',
})
ACCENT = '#58a6ff'
WARN   = '#d29922'
DANGER = '#f85149'
OK     = '#3fb950'
MUTED  = '#8b949e'

ZONES = {
    'Hadapsar Industrial':  (18.5018, 73.9320),
    'Pune NE Quadrant':     (18.5632, 73.9401),
    'Kothrud Residential':  (18.5074, 73.8077),
    'Katraj Hills':         (18.4524, 73.8567),
    'Shivajinagar':         (18.5308, 73.8474),
    'Aundh':                (18.5590, 73.8080),
}


## Section 1 — Architecture Diagram
The pipeline connects: [AQ (Real Data)] -> [WHO normalize] -> [Risk scorer] <-
  [LST (Pending Real Data / Simulated)] & [NDVI (Pending Real Data / Simulated)].

## Section 2 — AQ Sub-score Real Validation

In [ ]:
import pandas as pd
import numpy as np

WHO_BREAKPOINTS = [(0,15,0.0,0.25), (15,35,0.25,0.50), (35,75,0.50,0.75), (75,500,0.75,1.0)]

def who_normalize(pm25):
    for lo, hi, s_lo, s_hi in WHO_BREAKPOINTS:
        if lo <= pm25 < hi:
            return s_lo + (pm25-lo)/(hi-lo) * (s_hi-s_lo)
    return 1.0

def cpcb_normalize(pm25):
    if pm25 <= 30: return 'Good'
    elif pm25 <= 60: return 'Satisfactory'
    elif pm25 <= 90: return 'Moderate'
    elif pm25 <= 120: return 'Poor'
    else: return 'Very Poor'

df_aq = pd.read_sql_query("""
    SELECT station_name, value
    FROM raw_observations
    WHERE region_id = :region_id AND layer_type = 'aq'
      AND value > 0 AND value < 500
""", conn, params={"region_id": PUNE_REGION_ID})

from geopy.distance import geodesic
def get_nearest_zone(st_name):
    ST_COORDS = {'Savitribai Phule Pune University': (18.5308, 73.8473), 'Hadapsar': (18.4983, 73.9258), 'Katraj Dairy': (18.4500, 73.8650)}
    st_coord = ST_COORDS.get(st_name, (18.5308, 73.8473))
    best_zone = list(ZONES.keys())[0]
    min_dist = float('inf')
    for zone, coord in ZONES.items():
        dist = geodesic(st_coord, coord).km
        if dist < min_dist:
            min_dist = dist
            best_zone = zone
    return best_zone

df_aq['zone'] = df_aq['station_name'].apply(get_nearest_zone)
zone_aq = df_aq.groupby('zone')['value'].mean().to_dict()

print("=== Zone AQ Normalized Sub-scores ===")
tbl = []
for zone, pm25 in zone_aq.items():
    who_s = who_normalize(pm25)
    cpcb_cat = cpcb_normalize(pm25)
    tbl.append({
        'Zone': zone,
        'Mean PM2.5': f"{pm25:.2f}",
        'WHO Sub-score': f"{who_s:.3f}",
        'CPCB Index': cpcb_cat
    })
print(pd.DataFrame(tbl).to_markdown(index=False))


## Section 3 — Ablation Study

In [ ]:
plume_concs = {}
for zone in ZONES:
    val = conn.execute("""
        SELECT AVG(value) FROM ml_outputs
        WHERE model_type = 'gaussian_plume'
          AND zone_id = (SELECT id FROM zone_geometries WHERE name = :zone)
    """, {"zone": zone}).fetchone()[0]
    plume_concs[zone] = float(val) if val else 5.0

ablation_results = []
for zone in ZONES:
    pm = zone_aq.get(zone, 35.0)
    aq_score = who_normalize(pm)
    plume_conc = plume_concs.get(zone, 5.0)
    plume_norm = min(plume_conc / 100.0, 1.0)
    
    mock_LST = 35.0
    lst_norm = (mock_LST - 28) / (42 - 28)
    mock_NDVI = 0.15
    mock_ndvi_inv = 1.0 - mock_NDVI
    
    c1 = aq_score
    c2 = 0.6 * aq_score + 0.4 * plume_norm
    c3 = 0.5 * aq_score + 0.3 * lst_norm + 0.2 * mock_ndvi_inv
    c4 = 0.4 * aq_score + 0.3 * plume_norm + 0.2 * lst_norm + 0.1 * mock_ndvi_inv
    
    ablation_results.append({
        'Zone': zone,
        'C1: AQ': c1,
        'C2:+Plume': c2,
        'C3:+Mock': c3,
        'C4:Full': c4
    })

ab_df = pd.DataFrame(ablation_results)

diff_row = {'Zone': 'Score Range (max-min)'}
for col in ['C1: AQ', 'C2:+Plume', 'C3:+Mock', 'C4:Full']:
    diff_row[col] = ab_df[col].max() - ab_df[col].min()
    
ab_df = pd.concat([ab_df, pd.DataFrame([diff_row])], ignore_index=True)

print("=== Table 3: Ablation Study Comparison ===")
print(ab_df.to_markdown(index=False))


## Section 4 — KMeans Clustering Validation

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

X_cluster = np.array(list(zone_aq.values())).reshape(-1, 1)

results = []
for k in range(2, min(7, len(X_cluster))):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    if len(set(labels)) > 1:
        sil = silhouette_score(X_cluster, labels)
        db = davies_bouldin_score(X_cluster, labels)
        results.append({'k': k, 'silhouette': sil, 'db': db})
        print(f"k={k}: silhouette={sil:.3f}, DB={db:.3f}")

if results:
    opt_k = max(results, key=lambda x: x['silhouette'])['k']
    print(f"\nOptimal k: {opt_k}")
else:
    opt_k = 6

if results:
    plt.figure(figsize=(8, 5))
    ks = [r['k'] for r in results]
    sils = [r['silhouette'] for r in results]
    plt.plot(ks, sils, color=ACCENT, marker='o')
    plt.title('Silhouette Score vs. Cluster Count (k)', color='#c9d1d9')
    plt.xlabel('Cluster Count (k)')
    plt.ylabel('Silhouette Score')
    plt.grid(True)
    plt.savefig('outputs/05_silhouette.png')
    plt.show()


## Section 5 — LST Sensitivity Analysis

In [ ]:
n_sim = 1000
results_all = {zone: [] for zone in ZONES}

for _ in range(n_sim):
    base_lst = np.random.uniform(30, 42)
    for zone_name in ZONES:
        if 'Industrial' in zone_name or 'NE' in zone_name:
            zone_lst = base_lst + np.random.uniform(1, 4)
        elif 'Hills' in zone_name:
            zone_lst = base_lst - np.random.uniform(1, 3)
        else:
            zone_lst = base_lst + np.random.uniform(-1, 2)
            
        zone_lst = max(28, min(50, zone_lst))
        lst_norm = (zone_lst - 28) / (42 - 28)
        
        pm = zone_aq.get(zone_name, 35.0)
        aq_s = who_normalize(pm)
        plume_conc = plume_concs.get(zone_name, 5.0)
        plume_s = min(plume_conc / 100.0, 1.0)
        
        risk = 0.4 * aq_s + 0.3 * plume_s + 0.2 * lst_norm + 0.1 * (1 - 0.15)
        results_all[zone_name].append(risk)

print("=== Simulated Risk Score Bounds ===")
for zone, vals in results_all.items():
    print(f"{zone}: mean={np.mean(vals):.3f} (95% CI: [{np.percentile(vals, 2.5):.3f}, {np.percentile(vals, 97.5):.3f}])")

plt.figure(figsize=(10, 6))
plt.boxplot([results_all[z] for z in ZONES.keys()], vert=False)
plt.yticks(range(1, len(ZONES) + 1), ZONES.keys())
plt.title('Simulated Risk Scores with LST Uncertainty', color='#c9d1d9')
plt.xlabel('Risk Score')
plt.savefig('outputs/05_sensitivity.png')
plt.show()


## Section 6 — Paper Summary Table

In [ ]:
print("=== Summary of Risk Ablation and Sensitivity ===")
print(ab_df.to_markdown(index=False))
conn.close()
